In [5]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json 
import itertools

fsize = 20
plt.rcParams.update({"font.size": fsize})
%config InlineBackend.figure_format = 'retina'

In [26]:
def load_evidence(fn):
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    # Combine the original DataFrame with the unwrapped columns
    # df = pd.concat([df.drop(columns=['source', 'extracted', 'derived']), source_df, extracted_df, derived_df], axis=1)
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    del df["derived"]
    return df

In [135]:
fn_human = "../data/adipose_Massier2023/evidence_human/evidence.json"
# fn_human = "../data/adipose_Emont2022/evidence_human/evidence.json"

deg = load_evidence(fn_human)

In [136]:
deg.head()

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type
0,text,These included subtypes of (i) Th1-polarized (...,text,homo_sapiens,CD4+ T cells,White Adipose Tissue,none,CD4,gene
1,text,These included subtypes of (i) Th1-polarized (...,text,homo_sapiens,CD8+ T cells,White Adipose Tissue,none,CD8,gene
2,text,These included subtypes of (i) Th1-polarized (...,text,homo_sapiens,CD16+ NK (Natural Killer) Cells,White Adipose Tissue,none,CD16,gene
3,text,Top marker genes for these revealed that vC08 ...,text,homo_sapiens,myCO8 (M2-like macrophage),White Adipose Tissue,none,TIMP1,gene
4,text,Top marker genes for these revealed that vC08 ...,text,homo_sapiens,myCO8 (M2-like macrophage),White Adipose Tissue,none,ITLN1,gene


In [137]:
import re

def tokenize_with_offsets(text):
    """Tokenizes text and returns tokens with their character start positions."""
    tokens = []
    positions = []
    for match in re.finditer(r'\b\w+\b', text):  # Word-level tokenization
        tokens.append(match.group(0))
        positions.append((match.start(), match.start() + len(match.group(0))))  # Store start position in original text
    return tokens, positions

from collections import defaultdict
def index_source(source):
  source_tokens, source_positions = tokenize_with_offsets(source)
  source_index = defaultdict(list)
  for i, token in enumerate(source_tokens):
    source_index[token].append(source_positions[i])
  return source_index

def align_target(target, source_index):
  target_tokens, _ = tokenize_with_offsets(target)
  target_positions = []
  for token in target_tokens:
    if token in source_index.keys():
      target_positions.append(source_index[token].pop(0))
  return target_positions

def reconstruct_target(source, pos):
  return " ".join([source[i[0]: i[1]] for i in pos])

In [138]:
deg.head()

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type
0,text,These included subtypes of (i) Th1-polarized (...,text,homo_sapiens,CD4+ T cells,White Adipose Tissue,none,CD4,gene
1,text,These included subtypes of (i) Th1-polarized (...,text,homo_sapiens,CD8+ T cells,White Adipose Tissue,none,CD8,gene
2,text,These included subtypes of (i) Th1-polarized (...,text,homo_sapiens,CD16+ NK (Natural Killer) Cells,White Adipose Tissue,none,CD16,gene
3,text,Top marker genes for these revealed that vC08 ...,text,homo_sapiens,myCO8 (M2-like macrophage),White Adipose Tissue,none,TIMP1,gene
4,text,Top marker genes for these revealed that vC08 ...,text,homo_sapiens,myCO8 (M2-like macrophage),White Adipose Tissue,none,ITLN1,gene


In [139]:
tdeg = deg.query("source_type == 'text'").copy()

In [140]:
tdeg["len_extracted_cell_type_label"] = tdeg["extracted_cell_type_label"].apply(lambda x: len(x.split(" ")))
tdeg["pos_extracted_cell_type_label"] = tdeg.apply(
    lambda row: align_target(row["extracted_cell_type_label"], index_source(row["source_rationale"])), axis=1
)
tdeg["len_extracted_cell_type_label"] = tdeg["extracted_cell_type_label"].apply(lambda x: len(x.split(" ")))
tdeg["num_extracted_cell_type_label"] = tdeg.apply(
    lambda row: len(align_target(row["extracted_cell_type_label"], index_source(row["source_rationale"]))), axis=1
)
tdeg["found_extracted_cell_type"] = tdeg["len_extracted_cell_type_label"] == tdeg["num_extracted_cell_type_label"]

In [141]:
tdeg["len_extracted_feature_name"] = tdeg["extracted_feature_name"].apply(lambda x: len(x.split(" ")))
tdeg["pos_extracted_feature_name"] = tdeg.apply(
    lambda row: align_target(row["extracted_feature_name"], index_source(row["source_rationale"])), axis=1
)
tdeg["num_extracted_feature_name"] = tdeg.apply(
    lambda row: len(align_target(row["extracted_feature_name"], index_source(row["source_rationale"]))), axis=1
)
tdeg["found_extracted_feature_name"] = tdeg["len_extracted_feature_name"] == tdeg["num_extracted_feature_name"]


In [147]:
print("Genes found: , ", tdeg["found_extracted_feature_name"].sum())
print("Cell types found: , ", tdeg["found_extracted_cell_type"].sum())

Genes found: ,  29
Cell types found: ,  2


In [143]:
tdeg.shape

(29, 17)

In [144]:
tdeg

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type,len_extracted_cell_type_label,pos_extracted_cell_type_label,num_extracted_cell_type_label,found_extracted_cell_type,len_extracted_feature_name,pos_extracted_feature_name,num_extracted_feature_name,found_extracted_feature_name
0,text,These included subtypes of (i) Th1-polarized (...,text,homo_sapiens,CD4+ T cells,White Adipose Tissue,none,CD4,gene,3,"[(161, 164), (166, 167), (168, 173)]",3,True,1,"[(161, 164)]",1,True
1,text,These included subtypes of (i) Th1-polarized (...,text,homo_sapiens,CD8+ T cells,White Adipose Tissue,none,CD8,gene,3,"[(179, 182), (166, 167), (168, 173)]",3,True,1,"[(179, 182)]",1,True
2,text,These included subtypes of (i) Th1-polarized (...,text,homo_sapiens,CD16+ NK (Natural Killer) Cells,White Adipose Tissue,none,CD16,gene,5,"[(297, 301), (329, 331)]",2,False,1,"[(297, 301)]",1,True
3,text,Top marker genes for these revealed that vC08 ...,text,homo_sapiens,myCO8 (M2-like macrophage),White Adipose Tissue,none,TIMP1,gene,3,[],0,False,1,"[(263, 268)]",1,True
4,text,Top marker genes for these revealed that vC08 ...,text,homo_sapiens,myCO8 (M2-like macrophage),White Adipose Tissue,none,ITLN1,gene,3,[],0,False,1,"[(273, 278)]",1,True
5,text,Top marker genes for these revealed that vC08 ...,text,homo_sapiens,vC08 (blood vascular cells),White Adipose Tissue,none,TIMP1,gene,4,"[(41, 45), (170, 175)]",2,False,1,"[(263, 268)]",1,True
6,text,Top marker genes for these revealed that vC08 ...,text,homo_sapiens,vC08 (blood vascular cells),White Adipose Tissue,none,ITLN1,gene,4,"[(41, 45), (170, 175)]",2,False,1,"[(273, 278)]",1,True
7,text,vC09 was enriched for genes previously describ...,text,homo_sapiens,vC09 (blood vascular cells),White Adipose Tissue,none,TYROBP,gene,4,"[(0, 4), (82, 87)]",2,False,1,"[(96, 102)]",1,True
8,text,vC09 was enriched for genes previously describ...,text,homo_sapiens,vC09 (blood vascular cells),White Adipose Tissue,none,FCER1G,gene,4,"[(0, 4), (82, 87)]",2,False,1,"[(104, 110)]",1,True
9,text,One vascular subtype (vC05) expressed multiple...,text,homo_sapiens,vC05 (blood vascular cells),White Adipose Tissue,none,CXCL14,gene,4,"[(22, 26), (4, 12)]",2,False,1,"[(95, 101)]",1,True
